### Diabetes Prediction - Model Buiding and Evaluation

#### Introduction
This notebook focuses on developing and evaluating different ML classification models to predict the diabetes status.
For that the cleaned dataset will be reloaded and recreated all the preprocessing. This will be used to train multiple classification algorithms.

#### Objectives
- prepare the dataset for model building
- build multiple classification algorithms
- evalution will be conducted based on different metrices and compare the model
- select and save the best performing model

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [2]:
# reload dataset
data = pd.read_csv('../data/diabetes_cleaned.csv')
df3 = data.copy()
df3.head()

,age_group,gender,bmi,waist_circumference_cm,family_history_diabetes,family_history_obesity,family_history_hypertension,physical_activity_level,exercise_days_per_week,sedentary_hours_per_day,...,sleep_hours,stress_level,diet_quality_score,added_sugar_grams_per_day,fiber_grams_per_day,fruit_servings_per_day,systolic_bp,diastolic_bp,triglycerides_mg_dl,diabetes_status
0,31-40,Male,27.25,96.3,No,No,No,Extremely Active,7,6.6,...,6.3,Low,57.0,55.6,23.8,3.5,118.0,67.0,215.0,0.0
1,41-50,Female,31.47,88.5,Yes,No,Yes,Lightly Active,1,8.2,...,8.4,Moderate,51.3,79.3,4.5,1.4,118.0,80.0,203.2,0.0
2,51-60,Male,23.58,93.6,Yes,No,No,Lightly Active,1,5.4,...,5.1,Low,53.1,75.6,9.0,0.0,118.0,81.0,121.6,0.0
3,51-60,Female,23.62,71.3,Yes,No,Yes,Sedentary,1,6.2,...,6.1,High,53.8,30.1,11.5,3.3,120.0,83.0,106.5,1.0
4,51-60,Female,32.51,81.2,Yes,No,Yes,Lightly Active,0,6.2,...,6.7,Moderate,57.2,76.8,17.9,2.3,153.0,76.0,174.8,1.0


In [3]:
df3.shape

(20000, 21)

In [4]:
df3.isnull().sum()

age_group                      0
gender                         0
bmi                            0
waist_circumference_cm         0
family_history_diabetes        0
family_history_obesity         0
family_history_hypertension    0
physical_activity_level        0
exercise_days_per_week         0
sedentary_hours_per_day        0
smoking_status                 0
sleep_hours                    0
stress_level                   0
diet_quality_score             0
added_sugar_grams_per_day      0
fiber_grams_per_day            0
fruit_servings_per_day         0
systolic_bp                    0
diastolic_bp                   0
triglycerides_mg_dl            0
diabetes_status                0
dtype: int64

In [5]:
# separate features and target variable
X = df3.drop('diabetes_status', axis = 1)
y = df3['diabetes_status']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (20000, 20)
Target Shape: (20000,)


In [6]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

In [7]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (16000, 20)
X_test: (4000, 20)
y_train: (16000,)
y_test: (4000,)


In [8]:
# categorizing columns

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
ordinal_features = ['age_group', 'physical_activity_level', 'stress_level']
nominal_features = ['gender', 'smoking_status']
binary_features = ['family_history_diabetes', 'family_history_obesity', 'family_history_hypertension']


In [9]:
print("Numerical columns:", len(num_cols))
print("Ordinal columns:", len(ordinal_features))
print("Nominal columns:", len(nominal_features))
print("Binary columns:", len(binary_features))


Numerical columns: 12
Ordinal columns: 3
Nominal columns: 2
Binary columns: 3


In [10]:
# defining orinal categories
ordinal_categories = [
    ['18-30', '31-40', '41-50', '51-60', '61-70', '71-90'],
    ['Sedentary', 'Lightly Active', 'Moderately Active', 'Very Active', 'Extremely Active'],
    ['Low', 'Moderate', 'High', 'Very High']
]

In [11]:
# defining pipelines

numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

ordinal_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value= -1
    ))
])

nominal_pipeline = Pipeline([
    ('encoder', OneHotEncoder(
        drop = 'first',
        sparse_output = False, 
        handle_unknown='ignore' 
        
    ))
])

binary_pipeline = Pipeline([
    ('encoder', OrdinalEncoder(
        categories = [['No', 'Yes']] * 3 
    ))
    
])

In [12]:
# combining transformations using columntranformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, num_cols), 
        ('ordinal', ordinal_pipeline, ordinal_features),
        ('nominal', nominal_pipeline, nominal_features),
        ('binary', binary_pipeline, binary_features)
    ]
)

In [13]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nPercentage distribution:")
print(y_train.value_counts(normalize = True)*100)

Training target distribution:
diabetes_status
0.0    11525
1.0     4475
Name: count, dtype: int64

Percentage distribution:
diabetes_status
0.0    72.03125
1.0    27.96875
Name: proportion, dtype: float64


The target value shows a moderate class imbalance.

Here we are going to combine the preprocessing steps and LR classifier into a single pipeline.

### Model Building


#### 1. Logistic Regression
Logistic Regression is used as the baseline classification algorithm for predicting diabetes status

In [14]:
from sklearn.linear_model import LogisticRegression

#### Creation of LogisticRegression Pipeline (with class imbalance)
The preprocessing steps and LR classifier are combined into a single pipeline. This helps to reuse the same preprocessing during the prediction and deployment time.

In [15]:
logistic_pipeline = Pipeline([   # logistic_pipeline is the object
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000, # maxi. no. of iterations allowed for the optimization algorithm
        random_state = 42
    ))
])

#### Train the model

In [16]:
logistic_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](2,)","[0.,1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age_group','gender','bmi',...,'systolic_bp','diastolic_bp', 'triglycerides_mg_dl']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

Here we don't need to call fit_transform(). The pipeline automatically fits and transform the training data before training the estimator. During prediction it applies only the previously learned transformations.

#### ModelPrediction

In [17]:
y_pred = logistic_pipeline.predict(X_test)

In [18]:
print(y_pred[:10])

[0. 0. 0. 0. 1. 0. 0. 1. 0. 0.]


#### Model Evaluation
Model is evaluated using the evaluation metrices acuraccy, precision, recall, F1-score, ROC-AUC and confusion matrix

In [19]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score)


In [20]:
# evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Accuracy: 0.8145
Recall: 0.45933869526362825
Precision: 0.7895545314900153
F1 Score: 0.5807909604519774


In [21]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Confusion Matrix:
 [[2744  137]
 [ 605  514]]


In [22]:
print("Classification Report:\n", classification_report(y_test, y_pred))

Classification Report:
               precision    recall  f1-score   support

         0.0       0.82      0.95      0.88      2881
         1.0       0.79      0.46      0.58      1119

    accuracy                           0.81      4000
   macro avg       0.80      0.71      0.73      4000
weighted avg       0.81      0.81      0.80      4000



#### Logistic Regression with balanced class weights
To address class imbalance, LR is implemented wth class_weight='balanced'. This approch automatically assigns higher value to minority class

##### Creation of balanced model pipeline

In [23]:
logistic_balanced_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000, # maxi. no. of iterations allowed for the optimization algorithm
        class_weight='balanced',
        random_state=42
    ))
])

##### Train the model

In [24]:
logistic_balanced_pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](2,)","[0.,1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age_group','gender','bmi',...,'systolic_bp','diastolic_bp', 'triglycerides_mg_dl']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

##### Model prediction

In [25]:
y_pred_balanced = logistic_balanced_pipeline.predict(X_test)

In [26]:
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))
print("Recall:", recall_score(y_test, y_pred_balanced))
print("Precision:", precision_score(y_test, y_pred_balanced))
print("F1 Score:", f1_score(y_test, y_pred_balanced))

Accuracy: 0.722
Recall: 0.7211796246648794
Precision: 0.5021779713752333
F1 Score: 0.5920763022743947


In [27]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))


Confusion Matrix:
 [[2081  800]
 [ 312  807]]


In [28]:
#calculate ROC-AUC values

# for baseline logistic regression model
y_prob = logistic_pipeline.predict_proba(X_test)[:,1] #[:,1] ==> to get all rows and the all the probability of class 1
roc_auc = roc_auc_score(y_test, y_prob)
print("Baseline ROC-AUC:", roc_auc)

# for balanced logistic regression
y_prob_balanced = logistic_balanced_pipeline.predict_proba(X_test)[:,1]
roc_auc_balanced = roc_auc_score(y_test, y_prob_balanced)
print("Balanced ROC-AUC:", roc_auc_balanced)


Baseline ROC-AUC: 0.7977243900827554
Balanced ROC-AUC: 0.7975969023266981


##### Interpretation of baseline model
The baseline LR model achieved accuracy = 81.45%, precision = 78.96%, recall = 45.93%, F1-score = 58.08% and ROC-AUC = 0.7977.
The confusion matrix indicates that the model correctly identified 514 positive and 2744 negative cases. However, it incorrectly classified, 605 actual posituve cases as negative which results in low recall.

Although the balanced LR model improved recall to 72.12%. But it increased the false positives and reduced precision and accuracy. Both the models shows nearly identical ROC-AUC score.
The baseline LR model is retained as the initial benchmark model for further experimentation.

#### 2. Decision Tree Classifier
Decision Tree Classifier is our next classification algorithm to classify individuals into diabetes and non-diabetes categories. This algorithm focuses in improving classificarion perfeormance, in particuar, recall compared with LR.

In [29]:
from sklearn.tree import DecisionTreeClassifier

In [30]:
#pipeline creation
decision_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(
        random_state = 42,
    ))
])

In [31]:
# train the model
decision_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](2,)","[0.,1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age_group','gender','bmi',...,'systolic_bp','diastolic_bp', 'triglycerides_mg_dl']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [32]:
#prediction
y_pred_dt = decision_pipeline.predict(X_test)
y_pred_dt[:10]

array([0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])

In [33]:
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))

Accuracy: 0.69675
Recall: 0.4736371760500447
Precision: 0.4592720970537262
F1 Score: 0.4663440387153542


In [34]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))


Confusion Matrix:
 [[2257  624]
 [ 589  530]]


In [35]:
#roc- score
y_prob_dt = decision_pipeline.predict_proba(X_test)[:,1]
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)
print("ROC-AUC Score:", roc_auc_dt)


ROC-AUC Score: 0.6285228573759423


#### Interpretation of decision tree algorithm
The Decision Tree Classifier achieved an accuracy of 69.68%, recall of 47.36%, precision of 45.93%, F1-score of 46.63%, and ROC-AUC of 0.6285.

Compared with baseline Logistic Regression, Decision Tree achieved slightly higher recall but lower performance in accuracy, precision, F1-score, and ROC-AUC.
The Decision Tree model did not provide an overall improvement over baseline Logistic Regression.

#### 3. Random Forest Classifier

In [37]:
from sklearn.ensemble import RandomForestClassifier

In [38]:
#pipeline creation
random_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        random_state = 42,
    ))
])

In [39]:
#train the model 
random_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[float64](2,)","[0.,1.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](20,)","['age_group','gender','bmi',...,'systolic_bp','diastolic_bp', 'triglycerides_mg_dl']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,20
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [40]:
#predictions
y_pred_rf = random_pipeline.predict(X_test)

In [41]:
#evaluations
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))

Accuracy: 0.81025
Recall: 0.43699731903485256
Precision: 0.7912621359223301
F1 Score: 0.5630397236614854


In [42]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

Confusion Matrix:
 [[2752  129]
 [ 630  489]]


In [43]:
#roc-auc score
y_prob_rf = random_pipeline.predict_proba(X_test)[:,1]
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
print("ROC-AUC Score:", roc_auc_rf)

ROC-AUC Score: 0.7832290942568781


#### Interpretation
The Random Forest Classifier achieved an accuracy of 81.03%, recall of 43.70%, precision of 79.13%, F1-score of 56.30%, and ROC-AUC of 0.7832
The confusion matrix indicates that the model correctly identified 489 positive cases but incorrectly classified 630 actual positive cases as negative
The model achieved relatively high precision but showed lower recall.
It demonstrated better precision than the previous models but did not improve recall.